<a href="https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## SETUP

### 1. Install DuckD8

In [ ]:
!pip -q install duckdb

In [ ]:
import duckdb

con = duckdb.connect()

print("DuckDB connected:", con is not None)

DuckDB connected: True


### 2. Load HF_Token

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("FlyRank-ML")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [ ]:
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One raw row represents one client's content page performance observation for one report date. For this development slice, I use March 2026, covering 2026-03-01 through 2026-03-31.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

I will use historical search and engagement signals as features, including:
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_engaged_sessions`

These features describe a content page's observed performance and engagement before the decision moment.

### Label

The label is a provisional future-outcome proxy for Content Refresh Priority.
It is derived from April search-performance outcomes after the March decision
moment. Higher scores indicate greater review priority based on weaker future
search performance.

### Context

I will keep:
- `client_hash_id`
- `content_hash_id`
- `report_date`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

These fields identify the observation, its time context, and the availability of the underlying data sources.

### Excluded

I will exclude future performance outcomes from the feature set, such as future impressions, clicks, or position after the decision moment. I will also exclude identifier fields from model features because they identify entities rather than describe their performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. Query grain

In [ ]:
grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,9841378,9841378


2. Counts + windows/date span

In [ ]:
coverage_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

coverage_check

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


3. Availability + missing/available data

In [ ]:
availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_or_missing_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

availability_check

,total_rows,gsc_available_rows,gsc_unavailable_or_missing_rows
0,9841378,3611061,6230317


4. Feature frame

In [ ]:
feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
    WHERE month = '2026-03'
""").df()

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,<NA>


### Feature availability at the decision moment

- `gsc_impressions` — knowable at the decision moment because it is historical search-performance data available during the March reporting period.
- `gsc_clicks` — knowable at the decision moment because historical clicks are available during the March reporting period.
- `gsc_avg_position` — knowable at the decision moment because it summarizes observed search position during March.
- `ga4_pageviews` — knowable at the decision moment because historical pageviews are observed before the decision.
- `ga4_engaged_sessions` — knowable at the decision moment because historical engagement is observed before the decision.

In [ ]:
march_df = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

print("March rows:", len(march_df))
march_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 9841378


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>


In [ ]:
april_df = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
    )
""").df()

print("April rows:", len(april_df))
april_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April rows: 10424730


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,2026-04-01,9,0,54.777778
1,client_62f4a7e64f5e0096,content_13a8105125458098,2026-04-01,1,0,9.000000
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,2026-04-01,0,0,NaN
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,2026-04-01,1,0,7.000000
4,client_62f4a7e64f5e0096,content_ddbfb1907979759a,2026-04-01,0,0,NaN


In [ ]:
leakage_df = con.execute("""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_pageviews,
            ga4_engaged_sessions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
        )
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
        )
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,

        -- March = features available at decision time
        m.gsc_impressions AS impressions_march,
        m.gsc_clicks AS clicks_march,
        m.gsc_avg_position AS position_march,
        m.ga4_pageviews AS pageviews_march,
        m.ga4_engaged_sessions AS engaged_sessions_march,

        -- April = future outcome
        a.gsc_impressions AS impressions_april,
        a.gsc_clicks AS clicks_april,
        a.gsc_avg_position AS position_april

    FROM march m
    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id

    USING SAMPLE 50000
""").df()

print("Leakage experiment rows:", len(leakage_df))
leakage_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leakage experiment rows: 50000


,client_hash_id,content_hash_id,impressions_march,clicks_march,position_march,pageviews_march,engaged_sessions_march,impressions_april,clicks_april,position_april
0,client_3ffa76342f366962,content_60c1aaf28e5c5bd2,0,0,NaN,0,0,0,0,NaN
1,client_73cda7b4e4f265ea,content_72c218bf668b2377,128,0,2.007812,0,0,14,0,12.928571
2,client_62f4a7e64f5e0096,content_d8217e4409f594a0,225,2,5.346667,<NA>,<NA>,68,0,8.411765
3,client_c182d11e4862a37d,content_e0fa396f56471c5e,0,0,NaN,0,0,0,0,NaN
4,client_08a6a72ff48e62c0,content_2ebd2e032b8f4086,0,0,NaN,<NA>,<NA>,1,0,0.000000


### Future-outcome proxy label

For this demonstration, I construct a provisional Content Refresh Priority Score
from April search-performance outcomes observed after the March decision moment.
This proxy is used only for the leakage demonstration and is not treated as a
final production label.

Higher scores represent weaker future search performance and therefore higher
review priority.

In [ ]:
# Future-derived proxy label
# The target is intentionally constructed from April outcomes,
# which occur after the March decision moment.

leakage_df["future_ctr"] = (
    leakage_df["clicks_april"] /
    leakage_df["impressions_april"].replace(0, np.nan)
).fillna(0)

leakage_df["future_position"] = (
    leakage_df["position_april"].fillna(0)
)

leakage_df["content_refresh_priority_score"] = (
    (1 - leakage_df["future_ctr"].rank(pct=True))
    + leakage_df["future_position"].rank(pct=True)
) / 2

leakage_df[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "clicks_march",
        "position_march",
        "impressions_april",
        "clicks_april",
        "position_april",
        "content_refresh_priority_score"
    ]
].head(10)

,client_hash_id,content_hash_id,impressions_march,clicks_march,position_march,impressions_april,clicks_april,position_april,content_refresh_priority_score
0,client_3ffa76342f366962,content_60c1aaf28e5c5bd2,0,0,NaN,0,0,NaN,0.419210
1,client_73cda7b4e4f265ea,content_72c218bf668b2377,128,0,2.007812,14,0,12.928571,0.687715
2,client_62f4a7e64f5e0096,content_d8217e4409f594a0,225,2,5.346667,68,0,8.411765,0.661705
3,client_c182d11e4862a37d,content_e0fa396f56471c5e,0,0,NaN,0,0,NaN,0.419210
4,client_08a6a72ff48e62c0,content_2ebd2e032b8f4086,0,0,NaN,1,0,0.000000,0.419210
5,client_73cda7b4e4f265ea,content_5e0be289bdf20c23,6,0,15.666667,10,0,47.100000,0.740645
6,client_65de48885f4ef01b,content_9b1171594c3cc439,0,0,NaN,0,0,NaN,0.419210
7,client_62f4a7e64f5e0096,content_8ab54f81ab61b02a,463,0,2.842333,97,0,9.412371,0.670515
8,client_2910fd937f0b4d9a,content_70f54d875fad0fdd,0,0,NaN,0,0,NaN,0.419210
9,client_73cda7b4e4f265ea,content_de9fabe8994540e1,68,0,3.014706,52,0,2.865385,0.596735


### Deliberate label leakage

To demonstrate leakage, I intentionally add the target proxy itself as an input feature.

This is invalid because the Content Refresh Priority Score is the value the model is supposed to predict. At the decision moment, the model should not receive the answer as one of its inputs.

In [ ]:
# Deliberate label leakage:
# intentionally expose the future-derived target to the model as an input feature.

leakage_df["LEAKED_priority_score"] = (
    leakage_df["content_refresh_priority_score"]
)

leakage_df[
    [
        "impressions_march",
        "clicks_march",
        "position_march",
        "impressions_april",
        "clicks_april",
        "position_april",
        "LEAKED_priority_score",
        "content_refresh_priority_score"
    ]
].head(10)

,impressions_march,clicks_march,position_march,impressions_april,clicks_april,position_april,LEAKED_priority_score,content_refresh_priority_score
0,0,0,NaN,0,0,NaN,0.419210,0.419210
1,128,0,2.007812,14,0,12.928571,0.687715,0.687715
2,225,2,5.346667,68,0,8.411765,0.661705,0.661705
3,0,0,NaN,0,0,NaN,0.419210,0.419210
4,0,0,NaN,1,0,0.000000,0.419210,0.419210
5,6,0,15.666667,10,0,47.100000,0.740645,0.740645
6,0,0,NaN,0,0,NaN,0.419210,0.419210
7,463,0,2.842333,97,0,9.412371,0.670515,0.670515
8,0,0,NaN,0,0,NaN,0.419210,0.419210
9,68,0,3.014706,52,0,2.865385,0.596735,0.596735


### Leakage impact

I compare a model with the deliberately leaked target-derived feature against the same model after removing the leaked feature.

The leaked version should produce an unrealistically strong score because the model receives the target itself as an input.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Use a manageable sample for the leakage demonstration
demo_df = leakage_df.sample(
    n=min(50000, len(leakage_df)),
    random_state=42
).copy()

target_col = "content_refresh_priority_score"

# March = information available at decision moment
# LEAKED_priority_score = intentionally leaked future-derived target
leaky_features = [
    "impressions_march",
    "clicks_march",
    "position_march",
    "pageviews_march",
    "engaged_sessions_march",
    "LEAKED_priority_score"
]

# Prepare data
model_df = demo_df[
    leaky_features + [target_col]
].fillna(0)

X = model_df[leaky_features]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model_leaky = RandomForestRegressor(
    n_estimators=20,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model_leaky.fit(X_train, y_train)

y_pred = model_leaky.predict(X_test)

leaky_r2 = r2_score(y_test, y_pred)

print("Leaky model R²:", round(leaky_r2, 4))

Leaky model R²: 1.0


### Honest model after removing leakage

The leaked model achieved an unrealistically perfect R² of 1.0 because the target-derived feature was included as an input.

I therefore remove the leaked feature and train the same type of model using only features that could be known at the decision moment.

In [ ]:
# Remove the deliberately leaked target-derived feature.
# The honest model uses only information available
# at the March decision moment.

honest_features = [
    "impressions_march",
    "clicks_march",
    "position_march",
    "pageviews_march",
    "engaged_sessions_march"
]

honest_model_df = demo_df[
    honest_features + [target_col]
].copy()

honest_model_df = honest_model_df.fillna(0)

X_honest = honest_model_df[honest_features]
y_honest = honest_model_df[target_col]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_honest,
    y_honest,
    test_size=0.2,
    random_state=42
)

model_honest = RandomForestRegressor(
    n_estimators=20,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model_honest.fit(X_train_h, y_train_h)

y_pred_honest = model_honest.predict(X_test_h)

honest_r2 = r2_score(y_test_h, y_pred_honest)

print("Honest model R²:", round(honest_r2, 4))

Honest model R²: 0.4413


### Interpretation

The deliberately leaked model achieved an R² of 1.0 because the target-derived
`LEAKED_priority_score` was directly included as an input feature.

After removing the leaked feature, the honest model achieved an R² of 0.4413.
This substantial drop demonstrates that the perfect score of the leaky model
was caused by direct target leakage rather than genuine predictive capability.

The honest model uses only March information available at the decision moment,
while the proxy target is derived from April future outcomes.

This experiment demonstrates why future-derived target information must never be
included among model features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- The available history is not evenly balanced across all data sources. Some observations have GSC data available while others do not, so search-performance signals cannot be interpreted equally for every row.

- The March 2026 slice contains many rows where GSC data is unavailable or missing. In this slice, 3,611,061 rows have `gsc_data_available IS TRUE`, while 6,230,317 rows are unavailable or missing. Therefore, any model using GSC-based features may have a narrower usable population.

- This March slice describes observed performance during the reporting window, but it cannot by itself establish future outcomes or causal effects of making a content change. A future outcome window is needed to define and evaluate the label.

- Historical and future windows may overlap when constructing rolling features and outcomes. Care is needed to ensure that features only use information available at the decision moment and do not include future performance data.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.